# Семинар 4. Бенчмарк Yambda
Yambda-50M, Listen+ (прослушано ≥ 50%). Сравниваем DecayPop и SASRec по тестовому дню. Код официального бенчмарка, Colab + GPU.

Таргет — все прослушивания; каталог — все треки. С соседним ноутбуком метрики напрямую не сравниваем.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 1. Установка
Используем зависимости официального бенчмарка.

In [2]:
!pip uninstall -y -q polars
!pip install -q "polars-u64-idx==1.28.1" click optuna

In [3]:
import os, sys, json, time, subprocess, importlib.util
import numpy as np
import pandas as pd
import torch
from huggingface_hub import snapshot_download

ROOT = '/content/yambda'          # сюда скачаем код бенчмарка и данные
snapshot_download('yandex/yambda', repo_type='dataset', local_dir=ROOT,
                  allow_patterns=['benchmarks/*', 'flat/50m/listens.parquet', 'sequential/50m/listens.parquet'])
BENCH = f'{ROOT}/benchmarks'
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
EPOCHS = 100                      # как в бенчмарке; для пробного прогона поставьте 1–5
print(DEVICE, sorted(os.listdir(BENCH)))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 31 files:   0%|          | 0/31 [00:00<?, ?it/s]

cuda:0 ['.gitignore', 'Dockerfile', 'README.md', 'lint.sh', 'models', 'pyproject.toml', 'scripts', 'tests', 'yambda']


In [4]:
def run(model_dir, *args):
    # Запускает скрипт бенчмарка из его папки, как в README, и печатает вывод без прогресс-баров.
    env = {**os.environ, 'PYTHONPATH': BENCH}
    p = subprocess.Popen([sys.executable, *args], cwd=f'{BENCH}/models/{model_dir}', env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    out = []
    for line in p.stdout:
        out.append(line)
        if '%|' not in line:
            print(line, end='')
    assert p.wait() == 0, 'скрипт завершился с ошибкой'
    return ''.join(out)


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


sys.path[:0] = [BENCH, f'{BENCH}/models/sasrec']
import polars as pl
from yambda.constants import Constants
from yambda.evaluation.metrics import calc_metrics, create_target_mask
from yambda.evaluation.ranking import Embeddings, Ranked, Targets, rank_items
print('polars', pl.__version__)

polars 1.28.1


**Метрики:** NDCG@K (исправленный), HitRate@K, Recall@K, Coverage@K. Оставляем также NDCG из исходного кода для проверки.

In [5]:
KS = (10, 100)

def all_metrics(ranked, targets):
    theirs = calc_metrics(ranked, targets, [f'{m}@{k}' for m in ('ndcg', 'recall', 'coverage') for k in KS])
    mask = create_target_mask(ranked, targets)                                   # [users, 100]: 1 — попадание
    disc = 1 / torch.log2(torch.arange(2, mask.shape[1] + 2, device=mask.device, dtype=torch.float32))
    ideal = (torch.arange(mask.shape[1], device=mask.device)[None] < targets.lengths[:, None]).float()
    row = {}
    for k in KS:
        dcg, idcg = (mask[:, :k] * disc[:k]).sum(1), (ideal[:, :k] * disc[:k]).sum(1)
        row[f'NDCG@{k} (код)'] = theirs['ndcg'][k]
        row[f'HitRate@{k}'] = (mask[:, :k].sum(1) > 0).float().mean().item()
        row[f'NDCG@{k}'] = torch.where(idcg > 0, dcg / idcg, 0).mean().item()
        row[f'Recall@{k}'] = theirs['recall'][k]
        row[f'Coverage@{k}'] = theirs['coverage'][k]
    return row

RESULTS = {}

## 2. DecayPop
Затухающая популярность; параметры выбираем по val.

In [6]:
t0 = time.time()
out = run('popularity', 'main.py', '--data_dir', f'{ROOT}/flat', '--size', '50m', '--interaction', 'listens', '--device', DEVICE)
best_hour = float(out.split('best_hour=')[1].split()[0])
print(f'{time.time() - t0:.0f} с, лучшее окно затухания: {best_hour} ч')

REPORT METRICS: ('ndcg@10', 'ndcg@50', 'ndcg@100', 'dcg@10', 'dcg@50', 'dcg@100', 'recall@10', 'recall@50', 'recall@100', 'coverage@10', 'coverage@50', 'coverage@100')
SIZE 50m, INTERACTION listens







FINAL HYPERPARAMS best_hour=1.0

{
  "ndcg": {
    "10": 0.13437703251838684,
    "50": 0.2998477816581726,
    "100": 0.39030224084854126
  },
  "dcg": {
    "10": 0.10105513781309128,
    "50": 0.19294673204421997,
    "100": 0.25621598958969116
  },
  "recall": {
    "10": 0.022554680705070496,
    "50": 0.03314315155148506,
    "100": 0.04786578193306923
  },
  "coverage": {
    "10": 1.5890722678285963e-05,
    "50": 7.945361339142981e-05,
    "100": 0.00015890722678285962
  }
}
45 с, лучшее окно затухания: 1.0 ч


In [7]:
# Тот же финальный шаг, что в скрипте, но с нашим набором метрик
pop = load_module('popularity_main', f'{BENCH}/models/popularity/main.py')
train, _, test = pop.preprocess(pop.scan(f'{ROOT}/flat', '50m', 'listens'), 'listens', val_size=0)
max_ts = train.select(pl.col('timestamp').max()).collect(engine='streaming').item()
users = train.select('uid').unique().collect(engine='streaming')['uid'].to_torch().to(DEVICE)
emb = pop.training(hour=best_hour, train=train, max_timestamp=max_ts, device=DEVICE)
top = emb.ids[torch.topk(emb.embeddings, 100, dim=0).indices].ravel()
ranked = Ranked(user_ids=users, item_ids=top.expand((users.shape[0], 100)), num_item_ids=emb.ids.shape[0])
targets = Targets.from_sequential(test.group_by('uid', maintain_order=True).agg('item_id'), DEVICE)
RESULTS['DecayPop'] = all_metrics(ranked, targets)
print(f'тестовых пользователей: {len(targets):,}, треков: {emb.ids.shape[0]:,}')
pd.Series(RESULTS['DecayPop']).round(4)



Making target mask:   0%|          | 0/4599 [00:00<?, ?it/s]

Making target mask:   4%|▍         | 174/4599 [00:00<00:02, 1737.25it/s]

Making target mask:  11%|█         | 492/4599 [00:00<00:01, 2584.19it/s]

Making target mask:  19%|█▉        | 886/4599 [00:00<00:01, 3197.81it/s]

Making target mask:  28%|██▊       | 1310/4599 [00:00<00:00, 3608.60it/s]

Making target mask:  38%|███▊      | 1770/4599 [00:00<00:00, 3963.98it/s]

Making target mask:  48%|████▊     | 2191/4599 [00:00<00:00, 4044.93it/s]

Making target mask:  58%|█████▊    | 2652/4599 [00:00<00:00, 4227.11it/s]

Making target mask:  68%|██████▊   | 3142/4599 [00:00<00:00, 4437.22it/s]

Making target mask:  78%|███████▊  | 3590/4599 [00:00<00:00, 4450.32it/s]

Making target mask:  88%|████████▊ | 4065/4599 [00:01<00:00, 4542.16it/s]

Making target mask: 100%|██████████| 4599/4599 [00:01<00:00, 4113.28it/s]


Making target mask:   0%|          | 0/4599 [00:00<?, ?it/s]

Making target mask:  10%|█         | 471/4599 [00:0

тестовых пользователей: 4,599, треков: 629,298


,0
NDCG@10 (код),0.1344
HitRate@10,0.1344
NDCG@10,0.0245
Recall@10,0.0226
Coverage@10,0.0000
NDCG@100 (код),0.3903
HitRate@100,0.3903
NDCG@100,0.0312
Recall@100,0.0479
Coverage@100,0.0002


## 3. SASRec
Настройки: `max_len=200`, `d=64`, 2 слоя, 2 головы, batch=256, Adam `lr=1e-3`; прослушивания (`--interaction listens`).

In [8]:
t0 = time.time()
run('sasrec', 'train.py', '--exp_name', 'listens', '--data_dir', ROOT, '--interaction', 'listens',
    '--num_epochs', str(EPOCHS), '--device', DEVICE)
print(f'обучение: {(time.time() - t0) / 60:.1f} мин')

[2026-10-23 20:30:01] [DEBUG]: Preprocessing data...
[2026-10-23 20:30:22] [DEBUG]: Preprocessing data has finished!
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 3 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
[2026-10-23 20:30:28] [DEBUG]: Start training...
[2026-10-23 20:30:28] [DEBUG]: Start epoch 1
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 3 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker crea

Оценка: история train, таргет — прослушивания тестового дня, ранжирование по всему каталогу.

In [9]:
from data import EvalDataset, collate_fn, preprocess      # модули из benchmarks/models/sasrec
from model import SASRecEncoder                            # нужен, чтобы загрузить сохранённую модель
from torch.utils.data import DataLoader

data = preprocess(pl.scan_parquet(f'{ROOT}/sequential/50m/listens.parquet'), 'listens', val_size=0, max_seq_len=200)
train_df = data.train.collect(engine='streaming')
eval_df = (train_df.join(data.test.collect(engine='streaming'), on='uid', how='inner', suffix='_valid')
           .select('uid', pl.col('item_id').alias('item_id_train'), pl.col('item_id_valid')))
loader = DataLoader(EvalDataset(eval_df, max_seq_len=200), batch_size=256, collate_fn=collate_fn)

model = torch.load(f'{BENCH}/models/sasrec/checkpoints/listens_best_state.pth', weights_only=False).to(DEVICE).eval()
uids, user_emb = [], []
with torch.inference_mode():
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        uids.append(batch['user.ids']); user_emb.append(model(batch))
    items = model.item_embeddings.weight.data
    ranked = rank_items(Embeddings(torch.cat(uids), torch.cat(user_emb)),
                        Embeddings(torch.arange(items.shape[0], device=DEVICE), items), num_items=100)
targets = Targets(torch.tensor(eval_df['uid'].to_list(), device=DEVICE),
                  [torch.tensor(x, device=DEVICE) for x in eval_df['item_id_valid'].to_list()])
RESULTS['SASRec'] = all_metrics(ranked, targets)
print(f'тестовых пользователей: {len(targets):,}, треков: {data.num_items:,}')
pd.Series(RESULTS['SASRec']).round(4)



Calc topk by batches: 100%|██████████| 36/36 [00:00<00:00, 741.14it/s]


Making target mask:   0%|          | 0/4599 [00:00<?, ?it/s]

Making target mask:  11%|█         | 500/4599 [00:00<00:00, 4999.12it/s]

Making target mask:  22%|██▏       | 1000/4599 [00:00<00:00, 4953.68it/s]

Making target mask:  33%|███▎      | 1508/4599 [00:00<00:00, 5010.27it/s]

Making target mask:  44%|████▎     | 2010/4599 [00:00<00:00, 4871.37it/s]

Making target mask:  55%|█████▍    | 2521/4599 [00:00<00:00, 4955.41it/s]

Making target mask:  66%|██████▌   | 3018/4599 [00:00<00:00, 4521.44it/s]

Making target mask:  76%|███████▋  | 3516/4599 [00:00<00:00, 4660.60it/s]

Making target mask:  87%|████████▋ | 4014/4599 [00:00<00:00, 4755.25it/s]

Making target mask: 100%|██████████| 4599/4599 [00:00<00:00, 4767.15it/s]


Making target mask:   0%|          | 0/4599 [00:00<?, ?it/s]

Making target mask:  11%|█         | 497/4599 [00:00<00:00, 4961.75it/s]

Making target mask:  22%|██▏       | 994/4599 [00:00

тестовых пользователей: 4,599, треков: 631,003


,0
NDCG@10 (код),0.3153
HitRate@10,0.3153
NDCG@10,0.0742
Recall@10,0.0623
Coverage@10,0.0130
NDCG@100 (код),0.6327
HitRate@100,0.6327
NDCG@100,0.0803
Recall@100,0.1035
Coverage@100,0.0310


## 4. Результаты

In [10]:
table = pd.DataFrame(RESULTS).T
display(table[[c for c in table.columns if c.endswith('@10') or c.endswith('@10 (код)')]].round(4))
display(table[[c for c in table.columns if c.endswith('@100') or c.endswith('@100 (код)')]].round(4))

,NDCG@10 (код),HitRate@10,NDCG@10,Recall@10,Coverage@10
DecayPop,0.1344,0.1344,0.0245,0.0226,0.000
SASRec,0.3153,0.3153,0.0742,0.0623,0.013


,NDCG@100 (код),HitRate@100,NDCG@100,Recall@100,Coverage@100
DecayPop,0.3903,0.3903,0.0312,0.0479,0.0002
SASRec,0.6327,0.6327,0.0803,0.1035,0.0310


В исходном коде NDCG совпадает с HitRate из-за неверной нормировки. В выводах используем исправленный NDCG.

## Вывод

| Метрика | DecayPop | SASRec |
|---|---:|---:|
| NDCG@10 | 0.0245 | **0.0742** |
| HitRate@10 | 0.1344 | **0.3153** |
| Recall@10 | 0.0226 | **0.0623** |
| Coverage@10 | 0.0000 | **0.0130** |
| NDCG@100 | 0.0312 | **0.0803** |
| Recall@100 | 0.0479 | **0.1035** |

**SASRec лучше DecayPop по всем указанным метрикам.**